In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: Funkcja Qiskit od ColibriTD
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Funkcje Qiskit to eksperymentalna funkcjonalność dostępna dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (przez IBM Quantum Platform API). Są one w fazie wstępnego wydania i mogą ulec zmianie.
## Przegląd
Solver równań różniczkowych cząstkowych (PDE) przedstawiony tutaj jest częścią naszej platformy Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) i jest dostarczany jako Funkcja Qiskit. Za pomocą funkcji QUICK-PDE możesz rozwiązywać dziedzinowe równania różniczkowe cząstkowe na QPU IBM Quantum. Funkcja ta opiera się na algorytmie opisanym w [artykule ColibriTD dotyczącym H-DES](https://arxiv.org/abs/2410.01130). Algorytm ten potrafi rozwiązywać złożone problemy wielofizyczne, zaczynając od Obliczeniowej Dynamiki Płynów (CFD) i Deformacji Materiałów (MD), a kolejne zastosowania pojawią się wkrótce.

Aby zmierzyć się z równaniami różniczkowymi, rozwiązania próbne są kodowane jako liniowe kombinacje funkcji ortogonalnych (zazwyczaj wielomianów Czebyszewa, a dokładniej $2^n$ z nich, gdzie $n$ to liczba Qubitów kodujących twoją funkcję), parametryzowane kątami Zmiennego Obwodu Kwantowego (VQC). Ansatz generuje stan kodujący funkcję, która jest oceniana przez obserwable, których kombinacje pozwalają na wyznaczenie funkcji we wszystkich punktach. Możesz następnie ocenić funkcję straty, w której zakodowane są równania różniczkowe, i doprecyzować kąty w pętli hybrydowej, jak pokazano poniżej. Rozwiązania próbne stopniowo zbliżają się do rzeczywistych rozwiązań, aż uzyskasz zadowalający wynik.

![Przepływ pracy funkcji QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Oprócz tej pętli hybrydowej, możesz też łączyć ze sobą różne optymalizatory. Jest to przydatne, gdy chcesz, aby globalny optymalizator znalazł dobry zestaw kątów, a następnie bardziej precyzyjny optymalizator podążał gradientem do najlepszego zestawu sąsiednich kątów. W przypadku obliczeniowej dynamiki płynów (CFD) domyślna sekwencja optymalizacji daje najlepsze wyniki – natomiast w przypadku deformacji materiałów (MD), mimo że wartości domyślne zapewniają dobre rezultaty, możesz ją dalej konfigurować z korzyścią dla konkretnego problemu.

Zauważ, że dla każdej zmiennej funkcji określamy liczbę Qubitów (którą możesz modyfikować). Układając 10 identycznych Circuit i wyznaczając 10 identycznych obserwabli na różnych Qubitach w ramach jednego dużego Circuit, możesz ograniczać szum w procesie optymalizacji CMA, opierając się na metodzie uczenia szumu, i znacznie zmniejszyć potrzebną liczbę pomiarów.
### Obliczeniowa dynamika płynów
Bezlepkowe równanie Burgersa modeluje przepływ nielepkich płynów w następujący sposób:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ reprezentuje pole prędkości płynu. Ten przypadek użycia ma czasowy warunek brzegowy: możesz wybrać warunek początkowy, a następnie pozwolić systemowi się zrelaksować. Obecnie akceptowane są wyłącznie liniowe warunki początkowe: $ax + b$.

Argumenty równań różniczkowych CFD są na stałej siatce, jak następuje:

- $t$ jest z przedziału od 0 do 0,95 z 30 punktami próbkowania. $x$ jest z przedziału od 0 do 0,95 z krokiem 0,2375.

### Deformacja Materiałów
Ten przypadek użycia dotyczy hipoelastycznej deformacji przy jednowymiarowym teście rozciągania, w którym pręt zamocowany w przestrzeni jest ciągnięty na swoim drugim końcu. Problem opisujemy w następujący sposób:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ reprezentuje moduł objętościowy rozciąganego materiału, $n$ wykładnik prawa potęgowego, $b$ siłę na jednostkę masy, $\epsilon_0$ proporcjonalny limit naprężenia, $\sigma_0$ proporcjonalny limit odkształcenia, $u$ funkcję naprężenia, a $\sigma$ funkcję odkształcenia.

Rozważany pręt ma długość jednostkową. Ten przypadek użycia ma warunek brzegowy dla naprężenia powierzchniowego $t$, czyli ilości pracy potrzebnej do rozciągnięcia pręta.

Argumenty równań różniczkowych MD są na stałej siatce, jak następuje:

- $x$ jest z przedziału od 0 do 1 z krokiem 0,04.
## Testy porównawcze
Poniższa tabela przedstawia statystyki różnych przebiegów naszej funkcji.

| Przykład                           | Liczba Qubitów | Inicjalizacja         | Błąd      | Całkowity czas (min) | Użycie środowiska uruchomieniowego (min) |
| ---------------------------------- | -------------- | --------------------- | --------- | -------------------- | ---------------------------------------- |
| Bezlepkowe równanie Burgersa       | 50             | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                   | 25                                       |
| Hipoelastyczny 1D test rozciągania | 18             | `RANDOM`              | $10^{-2}$ | 123                  | 100                                      |
## Pierwsze kroki
Wypełnij [formularz, aby poprosić o dostęp do funkcji QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Następnie, zakładając że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w swoim lokalnym środowisku, wybierz funkcję w następujący sposób:

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_cloud / ibm_quantum_platform",
    instance="USER_CRN / HGP",
    token="USER_API_KEY / IQP_API_TOKEN",
)

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

In [ ]:
quick = catalog.load("colibritd/quick-pde")

Sprawdź [status](/guides/functions#check-job-status) swojego obciążenia Funkcji Qiskit lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(
    use_case="CFD_BURGER", physical_parameters={"a": 1.0, "b": 0.0}
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [ ]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())
solution = job.result()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_result_3d(result):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    t, x = np.meshgrid(result["samples"]["t"], result["samples"]["x"])

    ax.plot_surface(
        t,
        x,
        result["functions"]["u"],
        edgecolor="royalblue",
        lw=0.25,
        rstride=26,
        cstride=26,
        alpha=0.3,
    )
    ax.scatter(t, x, result["functions"]["u"], marker=".")
    ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

    plt.show()


# Call
plot_result_3d(solution)

![Wyjście poprzedniej komórki kodu](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Deformacja Materiałów
Przypadek użycia deformacji materiałów wymaga parametrów fizycznych materiału i przyłożonej siły, jak następuje:

In [ ]:
# Launches the solving for an arbitrary mu
job = quick.run(use_case="CFD_EULER", physical_parameters={"mu": 0.1})

solution = job.result()


# Colorplot function
def plot_result_2d(result):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    configs = {
        "g": {"cmap": "viridis", "title": "g(t, x)"},
        "u": {"cmap": "plasma", "title": "u(t, x)"},
    }

    t = result["samples"]["t"]
    x = result["samples"]["x"]

    for ax, (field, cfg) in zip(axes, configs.items()):
        v = result["functions"][field]

        im = ax.contourf(t, x, v, levels=50, cmap=cfg["cmap"])
        fig.colorbar(im, ax=ax, label=cfg["title"])

        ax.set_xlabel("t")
        ax.set_ylabel("x")
        ax.set_title(cfg["title"], fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()


plot_result_2d(solution)

![Wyjście poprzedniej komórki kodu](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

Poniżej przykład, jak uzyskać wartość funkcji dla konkretnego zestawu współrzędnych:

In [ ]:
# Select the properties of your material
job = quick.run(
    use_case="MD",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# Plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

## Pobieranie komunikatów o błędach
Jeśli status twojego obciążenia to `ERROR`, użyj `job.error_message()`, aby pobrać komunikat o błędzie pomocny przy debugowaniu, w następujący sposób:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Fetch error messages

If your workload status is `ERROR`, use `job.error_message()` to fetch the error message to help debug, as follows:

In [ ]:
job = quick.run(use_case="MD", physical_params={})

print(job.error_message())


# A wrapper can also be used for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

## Uzyskaj wsparcie

W celu uzyskania wsparcia skontaktuj się pod adresem qiskit-function-support@colibritd.com.

## Kolejne kroki

> **Tip:** - Wypełnij formularz, aby [poprosić o dostęp do funkcji QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK).
> - Odwiedź [dokumentację API](https://docs.quantum.ibm.com/api/functions/colibritd-pde) tej Funkcji Qiskit.
> - Wypróbuj modelowanie przepływu nielepkiego płynu za pomocą QUICK-PDE w [samouczku](/tutorials/colibritd-pde).
> - Zapoznaj się z [Jaffali, H., et al. (2025). H-DES: a Quantum-Classical Hybrid Differential Equation Solver. arXiv preprint arXiv:2410.01130](https://arxiv.org/abs/2410.01130).